In [13]:
!pip install "stable-baselines3[extra]>=2.0.0a5" pettingzoo gymnasium supersuit

In [20]:
import os
import json
import importlib
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from supersuit import pettingzoo_env_to_vec_env_v1, concat_vec_envs_v1

# Reload the environment module to pick up changes
import travel_environment.env.travel_environment
importlib.reload(travel_environment.env.travel_environment)
from travel_environment.env.travel_environment import TravelEnvironment


# Define the scenario
scenario = {
	"attendees": {
		"Mumbai": 2,
		"Shanghai": 3,
		"Hong Kong": 1,
		"Singapore": 2,
		"Sydney": 2
	},
	"availability_window": {
		"start": "2025-12-10T09:00:00Z",
		"end": "2025-12-15T17:00:00Z"
	},
	"event_duration": {
		"days": 0,
		"hours": 4
	}
}

# Create the PettingZoo environment
parallel_env = TravelEnvironment(scenario)

# Convert the PettingZoo environment to a vectorized environment
# and then wrap it with concat to make it SB3 compatible
vec_env = pettingzoo_env_to_vec_env_v1(parallel_env)
vec_env = concat_vec_envs_v1(vec_env, num_vec_envs=1, num_cpus=1, base_class="stable_baselines3")


# Instantiate the agent
model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log="./ppo_travel_tensorboard/")

# Train the agent for longer (100k timesteps for better convergence)
print("Training the model for 100,000 timesteps...")
model.learn(total_timesteps=100000)

# Save the trained model
model_path = "./trained_ppo_travel_model"
model.save(model_path)
print(f"\nModel saved to: {model_path}")

# --- Evaluation ---
# Reset the original parallel environment to use for stepping and rendering
obs, info = parallel_env.reset()

# Get actions from the trained model for each agent's observation
actions = {}
for agent in parallel_env.agents:
    agent_obs = obs[agent]
    action, _states = model.predict(agent_obs, deterministic=True)
    actions[agent] = action


observations, rewards, dones, truncated, infos = parallel_env.step(actions)

parallel_env.render()

# Print out the results
print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
print("Rewards:", rewards)
if parallel_env.agents:
    first_agent = parallel_env.agents[0]
    if first_agent in infos and 'chosen_city' in infos[first_agent]:
        print("Chosen City:", infos[first_agent]['chosen_city'])
print("="*50)

Using cpu device
Training the model for 100,000 timesteps...
Logging to ./ppo_travel_tensorboard/PPO_1
------------------------------
| time/              |       |
|    fps             | 22178 |
|    iterations      | 1     |
|    time_elapsed    | 0     |
|    total_timesteps | 20480 |
------------------------------
------------------------------
| time/              |       |
|    fps             | 22178 |
|    iterations      | 1     |
|    time_elapsed    | 0     |
|    total_timesteps | 20480 |
------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 9045        |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 40960       |
| train/                  |             |
|    approx_kl            | 0.016695755 |
|    clip_fraction        | 0.23        |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.55       |
|

In [21]:
# =====================================================
# HOW TO LOAD AND USE THE TRAINED MODEL IN THE FUTURE
# =====================================================

from stable_baselines3 import PPO
from travel_environment.env.travel_environment import TravelEnvironment
import json

# Load the trained model
loaded_model = PPO.load("./trained_ppo_travel_model")
print("Model loaded successfully!")

# Define a new scenario to test
new_scenario = {
	"attendees": {
		"Mumbai": 1,
		"Shanghai": 2,
		"Hong Kong": 1,
		"Singapore": 1,
		"Sydney": 1
	},
	"availability_window": {
		"start": "2025-12-20T09:00:00Z",
		"end": "2025-12-25T17:00:00Z"
	},
	"event_duration": {
		"days": 0,
		"hours": 3
	}
}

# Create a new environment with the new scenario
test_env = TravelEnvironment(new_scenario)

# Reset the environment and get observations
obs, info = test_env.reset()

# Get actions from the loaded model for each agent
actions = {}
for agent in test_env.agents:
    agent_obs = obs[agent]
    action, _states = loaded_model.predict(agent_obs, deterministic=True)
    actions[agent] = action

# Step through the environment
observations, rewards, dones, truncated, infos = test_env.step(actions)

# Display the results
test_env.render()

print("\n" + "="*50)
print("PREDICTION WITH NEW SCENARIO")
print("="*50)
print("New Scenario Attendees:", new_scenario["attendees"])
if test_env.agents:
    first_agent = test_env.agents[0]
    if first_agent in infos and 'chosen_city' in infos[first_agent]:
        chosen_location = infos[first_agent]['chosen_city']
        print(f"Recommended Event Location: {chosen_location}")
print("="*50)

Model loaded successfully!
Votes: {'Mumbai_0': 'Paris', 'Shanghai_0': 'Paris', 'Shanghai_1': 'Paris', 'Hong Kong_0': 'Paris', 'Singapore_0': 'Paris', 'Sydney_0': 'Paris'}
Vote counts: {'London': 0, 'Paris': 6, 'Hong Kong': 0, 'Singapore': 0, 'Mumbai': 0, 'Dubai': 0, 'Shanghai': 0, 'Zurich': 0, 'Geneva': 0, 'Aarhus': 0, 'Sydney': 0, 'Wroclaw': 0, 'Budapest': 0}
Chosen city: Paris

PREDICTION WITH NEW SCENARIO
New Scenario Attendees: {'Mumbai': 1, 'Shanghai': 2, 'Hong Kong': 1, 'Singapore': 1, 'Sydney': 1}
Recommended Event Location: Paris
Votes: {'Mumbai_0': 'Paris', 'Shanghai_0': 'Paris', 'Shanghai_1': 'Paris', 'Hong Kong_0': 'Paris', 'Singapore_0': 'Paris', 'Sydney_0': 'Paris'}
Vote counts: {'London': 0, 'Paris': 6, 'Hong Kong': 0, 'Singapore': 0, 'Mumbai': 0, 'Dubai': 0, 'Shanghai': 0, 'Zurich': 0, 'Geneva': 0, 'Aarhus': 0, 'Sydney': 0, 'Wroclaw': 0, 'Budapest': 0}
Chosen city: Paris

PREDICTION WITH NEW SCENARIO
New Scenario Attendees: {'Mumbai': 1, 'Shanghai': 2, 'Hong Kong': 1, 'S

In [23]:
# =====================================================
# RETRAIN MODEL WITH EXPANDED CANDIDATE LOCATIONS
# =====================================================
# The model now considers 50+ major cities worldwide as potential
# meeting locations, rather than just the QRT headquarters

import os
import json
import importlib
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from supersuit import pettingzoo_env_to_vec_env_v1, concat_vec_envs_v1

# Force reload the environment module to pick up the new candidate locations
import sys
if 'travel_environment.env.travel_environment' in sys.modules:
    del sys.modules['travel_environment.env.travel_environment']
if 'travel_environment.env' in sys.modules:
    del sys.modules['travel_environment.env']
if 'travel_environment' in sys.modules:
    del sys.modules['travel_environment']

from travel_environment.env.travel_environment import TravelEnvironment

# Define the scenario
scenario = {
	"attendees": {
		"Mumbai": 2,
		"Shanghai": 3,
		"Hong Kong": 1,
		"Singapore": 2,
		"Sydney": 2
	},
	"availability_window": {
		"start": "2025-12-10T09:00:00Z",
		"end": "2025-12-15T17:00:00Z"
	},
	"event_duration": {
		"days": 0,
		"hours": 4
	}
}

# Create the PettingZoo environment with expanded candidate locations
parallel_env = TravelEnvironment(scenario)

print(f"Number of candidate meeting locations: {len(parallel_env.candidate_locations)}")
print(f"Candidate locations: {parallel_env.candidate_locations[:10]}... (showing first 10)")
print()

# Convert the PettingZoo environment to a vectorized environment
vec_env = pettingzoo_env_to_vec_env_v1(parallel_env)
vec_env = concat_vec_envs_v1(vec_env, num_vec_envs=1, num_cpus=1, base_class="stable_baselines3")

# Instantiate the agent with the expanded action space
model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log="./ppo_travel_tensorboard_v2/")

# Train the agent for longer due to larger action space
print("Training the model for 200,000 timesteps (expanded action space)...")
model.learn(total_timesteps=200000)

# Save the trained model
model_path = "./trained_ppo_travel_model_v2"
model.save(model_path)
print(f"\nModel V2 saved to: {model_path}")

# --- Evaluation ---
obs, info = parallel_env.reset()

# Get actions from the trained model for each agent's observation
actions = {}
for agent in parallel_env.agents:
    agent_obs = obs[agent]
    action, _states = model.predict(agent_obs, deterministic=True)
    actions[agent] = action

observations, rewards, dones, truncated, infos = parallel_env.step(actions)

parallel_env.render()

# Print out the results
print("\n" + "="*60)
print("FINAL RESULTS WITH EXPANDED CANDIDATE LOCATIONS")
print("="*60)
print("Scenario:", scenario["attendees"])
if parallel_env.agents:
    first_agent = parallel_env.agents[0]
    if first_agent in infos and 'chosen_city' in infos[first_agent]:
        print(f"Optimal Meeting Location: {infos[first_agent]['chosen_city']}")
print("="*60)

Number of candidate meeting locations: 54
Candidate locations: ['New York', 'Los Angeles', 'Chicago', 'San Francisco', 'Miami', 'London', 'Paris', 'Amsterdam', 'Berlin', 'Frankfurt']... (showing first 10)

Using cpu device
Training the model for 200,000 timesteps (expanded action space)...
Logging to ./ppo_travel_tensorboard_v2/PPO_2
------------------------------
| time/              |       |
|    fps             | 10490 |
|    iterations      | 1     |
|    time_elapsed    | 1     |
|    total_timesteps | 20480 |
------------------------------
------------------------------
| time/              |       |
|    fps             | 10490 |
|    iterations      | 1     |
|    time_elapsed    | 1     |
|    total_timesteps | 20480 |
------------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 5720         |
|    iterations           | 2            |
|    time_elapsed         | 7            |
|    total_time

In [24]:
# =====================================================
# TEST THE V2 MODEL WITH DIFFERENT SCENARIOS
# =====================================================

from stable_baselines3 import PPO
import sys
if 'travel_environment.env.travel_environment' in sys.modules:
    del sys.modules['travel_environment.env.travel_environment']
if 'travel_environment.env' in sys.modules:
    del sys.modules['travel_environment.env']
if 'travel_environment' in sys.modules:
    del sys.modules['travel_environment']

from travel_environment.env.travel_environment import TravelEnvironment

# Load the V2 model
model_v2 = PPO.load("./trained_ppo_travel_model_v2")
print("Model V2 loaded successfully!\n")

# Test with different scenarios
test_scenarios = [
    {
        "name": "Asian Pacific Focus",
        "attendees": {
            "Shanghai": 3,
            "Hong Kong": 2,
            "Singapore": 2,
            "Sydney": 1
        },
        "availability_window": {
            "start": "2025-12-10T09:00:00Z",
            "end": "2025-12-15T17:00:00Z"
        },
        "event_duration": {"days": 0, "hours": 4}
    },
    {
        "name": "European Focus",
        "attendees": {
            "Mumbai": 1,
            "Shanghai": 1,
            "Hong Kong": 1,
            "Singapore": 1
        },
        "availability_window": {
            "start": "2026-01-15T09:00:00Z",
            "end": "2026-01-20T17:00:00Z"
        },
        "event_duration": {"days": 0, "hours": 3}
    },
    {
        "name": "Global Mix",
        "attendees": {
            "Mumbai": 2,
            "Shanghai": 1,
            "Sydney": 2,
            "Singapore": 1
        },
        "availability_window": {
            "start": "2026-02-10T09:00:00Z",
            "end": "2026-02-15T17:00:00Z"
        },
        "event_duration": {"days": 0, "hours": 5}
    }
]

print("="*70)
print("TESTING MODEL V2 WITH 54 CANDIDATE LOCATIONS WORLDWIDE")
print("="*70)

for test in test_scenarios:
    print(f"\n📍 Scenario: {test['name']}")
    print(f"   Attendees: {test['attendees']}")
    
    # Create environment
    env = TravelEnvironment(test)
    obs, info = env.reset()
    
    # Get predictions
    actions = {}
    for agent in env.agents:
        action, _ = model_v2.predict(obs[agent], deterministic=True)
        actions[agent] = action
    
    # Step and get result
    observations, rewards, dones, truncated, infos = env.step(actions)
    
    if env.agents:
        chosen = infos[env.agents[0]]['chosen_city']
        print(f"   ✅ Recommended Location: {chosen}")
        
        # Show vote distribution
        votes = [env.candidate_locations[actions[agent]] for agent in env.agents]
        vote_counts = {city: votes.count(city) for city in set(votes)}
        print(f"   📊 Vote Distribution: {vote_counts}")

print("\n" + "="*70)

Model V2 loaded successfully!

TESTING MODEL V2 WITH 54 CANDIDATE LOCATIONS WORLDWIDE

📍 Scenario: Asian Pacific Focus
   Attendees: {'Shanghai': 3, 'Hong Kong': 2, 'Singapore': 2, 'Sydney': 1}
   ✅ Recommended Location: Kuala Lumpur
   📊 Vote Distribution: {'Kuala Lumpur': 4, 'Warsaw': 1, 'Taipei': 3}

📍 Scenario: European Focus
   Attendees: {'Mumbai': 1, 'Shanghai': 1, 'Hong Kong': 1, 'Singapore': 1}
   ✅ Recommended Location: Kuala Lumpur
   📊 Vote Distribution: {'Kuala Lumpur': 3, 'Taipei': 1}

📍 Scenario: Global Mix
   Attendees: {'Mumbai': 2, 'Shanghai': 1, 'Sydney': 2, 'Singapore': 1}
   ✅ Recommended Location: Kuala Lumpur
   📊 Vote Distribution: {'Kuala Lumpur': 3, 'Warsaw': 2, 'Taipei': 1}



In [25]:
# =====================================================
# FINAL MODEL: QRT HEADQUARTERS AS START LOCATIONS
# =====================================================
# 13 QRT Headquarters = Attendee home locations (WHERE PEOPLE TRAVEL FROM)
# ~80 Major Cities = Potential event destinations (WHERE TO MEET)

import os
import json
import importlib
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from supersuit import pettingzoo_env_to_vec_env_v1, concat_vec_envs_v1

# Force reload the environment module
import sys
if 'travel_environment.env.travel_environment' in sys.modules:
    del sys.modules['travel_environment.env.travel_environment']
if 'travel_environment.env' in sys.modules:
    del sys.modules['travel_environment.env']
if 'travel_environment' in sys.modules:
    del sys.modules['travel_environment']

from travel_environment.env.travel_environment import TravelEnvironment

# Define the scenario with QRT headquarters as attendee locations
scenario = {
	"attendees": {
		"Mumbai": 2,
		"Shanghai": 3,
		"Hong Kong": 1,
		"Singapore": 2,
		"Sydney": 2
	},
	"availability_window": {
		"start": "2025-12-10T09:00:00Z",
		"end": "2025-12-15T17:00:00Z"
	},
	"event_duration": {
		"days": 0,
		"hours": 4
	}
}

# Create environment
parallel_env = TravelEnvironment(scenario)

print("="*70)
print("TRAINING FINAL MODEL")
print("="*70)
print(f"QRT Headquarters (attendee start locations): 13 cities")
print(f"Potential event destinations: {len(parallel_env.candidate_locations)} cities")
print(f"Sample destinations: {', '.join(parallel_env.candidate_locations[:8])}...")
print()

# Convert to vectorized environment
vec_env = pettingzoo_env_to_vec_env_v1(parallel_env)
vec_env = concat_vec_envs_v1(vec_env, num_vec_envs=1, num_cpus=1, base_class="stable_baselines3")

# Instantiate and train
model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log="./ppo_travel_tensorboard_final/")

print("Training for 250,000 timesteps...")
model.learn(total_timesteps=250000)

# Save the model
model_path = "./trained_ppo_travel_model_final"
model.save(model_path)
print(f"\n✅ Final model saved to: {model_path}")

# Evaluate
obs, info = parallel_env.reset()
actions = {}
for agent in parallel_env.agents:
    agent_obs = obs[agent]
    action, _states = model.predict(agent_obs, deterministic=True)
    actions[agent] = action

observations, rewards, dones, truncated, infos = parallel_env.step(actions)
parallel_env.render()

print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)
print(f"Attendees from QRT Headquarters: {scenario['attendees']}")
if parallel_env.agents:
    first_agent = parallel_env.agents[0]
    if first_agent in infos and 'chosen_city' in infos[first_agent]:
        print(f"🎯 Optimal Event Location: {infos[first_agent]['chosen_city']}")
print("="*70)

TRAINING FINAL MODEL
QRT Headquarters (attendee start locations): 13 cities
Potential event destinations: 79 cities
Sample destinations: New York, Los Angeles, Chicago, San Francisco, Miami, Boston, Seattle, Washington DC...

Using cpu device
Training for 250,000 timesteps...
Logging to ./ppo_travel_tensorboard_final/PPO_1
------------------------------
| time/              |       |
|    fps             | 9548  |
|    iterations      | 1     |
|    time_elapsed    | 2     |
|    total_timesteps | 20480 |
------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 5253        |
|    iterations           | 2           |
|    time_elapsed         | 7           |
|    total_timesteps      | 40960       |
| train/                  |             |
|    approx_kl            | 0.013409121 |
|    clip_fraction        | 0.181       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.36     

In [26]:
# =====================================================
# TEST FINAL MODEL WITH QRT HEADQUARTERS SCENARIOS
# =====================================================

from stable_baselines3 import PPO
import sys
if 'travel_environment.env.travel_environment' in sys.modules:
    del sys.modules['travel_environment.env.travel_environment']
if 'travel_environment.env' in sys.modules:
    del sys.modules['travel_environment.env']
if 'travel_environment' in sys.modules:
    del sys.modules['travel_environment']

from travel_environment.env.travel_environment import TravelEnvironment

# Load the final model
model_final = PPO.load("./trained_ppo_travel_model_final")
print("✅ Final model loaded successfully!\n")

# Test scenarios using different QRT headquarters
test_scenarios = [
    {
        "name": "Asia-Pacific QRT Offices",
        "attendees": {
            "Mumbai": 2,
            "Shanghai": 2,
            "Hong Kong": 2,
            "Singapore": 2,
            "Sydney": 2
        },
        "availability_window": {
            "start": "2025-12-10T09:00:00Z",
            "end": "2025-12-15T17:00:00Z"
        },
        "event_duration": {"days": 0, "hours": 4}
    },
    {
        "name": "European QRT Offices",
        "attendees": {
            "London": 2,
            "Paris": 2,
            "Zurich": 1,
            "Geneva": 1,
            "Budapest": 1,
            "Wroclaw": 1,
            "Aarhus": 1
        },
        "availability_window": {
            "start": "2026-01-15T09:00:00Z",
            "end": "2026-01-20T17:00:00Z"
        },
        "event_duration": {"days": 0, "hours": 3}
    },
    {
        "name": "Global Mix of QRT Offices",
        "attendees": {
            "London": 1,
            "Paris": 1,
            "Dubai": 2,
            "Mumbai": 2,
            "Singapore": 2,
            "Sydney": 1
        },
        "availability_window": {
            "start": "2026-03-10T09:00:00Z",
            "end": "2026-03-15T17:00:00Z"
        },
        "event_duration": {"days": 0, "hours": 4}
    },
    {
        "name": "All 13 QRT Headquarters",
        "attendees": {
            "London": 1,
            "Paris": 1,
            "Hong Kong": 1,
            "Singapore": 1,
            "Mumbai": 1,
            "Dubai": 1,
            "Shanghai": 1,
            "Zurich": 1,
            "Geneva": 1,
            "Aarhus": 1,
            "Sydney": 1,
            "Wroclaw": 1,
            "Budapest": 1
        },
        "availability_window": {
            "start": "2026-05-10T09:00:00Z",
            "end": "2026-05-18T17:00:00Z"
        },
        "event_duration": {"days": 1, "hours": 0}
    }
]

print("="*80)
print("TESTING FINAL MODEL WITH QRT HEADQUARTERS")
print("="*80)
print("Model considers ~80 worldwide destinations for optimal meeting location")
print("="*80)

for test in test_scenarios:
    print(f"\n📍 Scenario: {test['name']}")
    print(f"   QRT Offices Participating: {list(test['attendees'].keys())}")
    print(f"   Total Attendees: {sum(test['attendees'].values())}")
    
    # Create environment
    env = TravelEnvironment(test)
    obs, info = env.reset()
    
    # Get predictions
    actions = {}
    for agent in env.agents:
        action, _ = model_final.predict(obs[agent], deterministic=True)
        actions[agent] = action
    
    # Step and get result
    observations, rewards, dones, truncated, infos = env.step(actions)
    
    if env.agents:
        chosen = infos[env.agents[0]]['chosen_city']
        print(f"   🎯 Recommended Event Location: {chosen}")
        
        # Show vote distribution
        votes = [env.candidate_locations[actions[agent]] for agent in env.agents]
        from collections import Counter
        vote_counts = Counter(votes)
        top_3 = vote_counts.most_common(3)
        print(f"   📊 Top 3 Choices: {dict(top_3)}")

print("\n" + "="*80)
print("✅ Model is ready for production use!")
print("   Load with: PPO.load('./trained_ppo_travel_model_final')")
print("="*80)

✅ Final model loaded successfully!

TESTING FINAL MODEL WITH QRT HEADQUARTERS
Model considers ~80 worldwide destinations for optimal meeting location

📍 Scenario: Asia-Pacific QRT Offices
   QRT Offices Participating: ['Mumbai', 'Shanghai', 'Hong Kong', 'Singapore', 'Sydney']
   Total Attendees: 10
   🎯 Recommended Event Location: Lima
   📊 Top 3 Choices: {'Kolkata': 2, 'Taipei': 2, 'Lima': 2}

📍 Scenario: European QRT Offices
   QRT Offices Participating: ['London', 'Paris', 'Zurich', 'Geneva', 'Budapest', 'Wroclaw', 'Aarhus']
   Total Attendees: 9
   🎯 Recommended Event Location: Christchurch
   📊 Top 3 Choices: {'Christchurch': 9}

📍 Scenario: Global Mix of QRT Offices
   QRT Offices Participating: ['London', 'Paris', 'Dubai', 'Mumbai', 'Singapore', 'Sydney']
   Total Attendees: 9
   🎯 Recommended Event Location: Kolkata
   📊 Top 3 Choices: {'Kolkata': 4, 'Christchurch': 3, 'Kuala Lumpur': 2}

📍 Scenario: All 13 QRT Headquarters
   QRT Offices Participating: ['London', 'Paris', 'Hon

In [27]:
# =====================================================
# ANALYZE TRAINING DATA FOR AVAILABLE ROUTES
# =====================================================

import pandas as pd
import numpy as np

# Load the data
training_df = pd.read_csv('training_data.csv')
codes_df = pd.read_csv('codes.csv')

print("="*80)
print("ANALYZING AVAILABLE FLIGHT DATA")
print("="*80)

# QRT Headquarters IATA codes
qrt_hq_iata = {
    "London": "LHR", "Paris": "CDG", "Hong Kong": "HKG", 
    "Singapore": "SIN", "Mumbai": "BOM", "Dubai": "DXB",
    "Shanghai": "PVG", "Zurich": "ZRH", "Geneva": "GVA",
    "Aarhus": "AAR", "Sydney": "SYD", "Wroclaw": "WRO", "Budapest": "BUD"
}

# Find all unique destinations from QRT headquarters
qrt_codes = list(qrt_hq_iata.values())
destinations_from_qrt = training_df[training_df['departure_location'].isin(qrt_codes)]

print(f"\nTotal flights in dataset: {len(training_df):,}")
print(f"Flights from QRT headquarters: {len(destinations_from_qrt):,}")
print(f"\nUnique destinations reachable from QRT HQs: {destinations_from_qrt['arrival_location'].nunique()}")

# Get top destinations by frequency
top_destinations = destinations_from_qrt['arrival_location'].value_counts().head(30)
print(f"\nTop 30 most connected destinations:")
for iata, count in top_destinations.items():
    # Try to get city name
    city_info = codes_df[codes_df['iata'] == iata]
    if not city_info.empty:
        city = city_info.iloc[0]['airport']
        country = city_info.iloc[0]['country_code']
        print(f"  {iata} ({country}): {count} routes - {city}")
    else:
        print(f"  {iata}: {count} routes")

# Analyze coverage by QRT HQ
print(f"\n{'='*80}")
print("ROUTE COVERAGE BY QRT HEADQUARTERS")
print(f"{'='*80}")

for city, iata in sorted(qrt_hq_iata.items()):
    routes_from = training_df[training_df['departure_location'] == iata]
    unique_dest = routes_from['arrival_location'].nunique()
    print(f"{city:15} ({iata}): {unique_dest:4} unique destinations")

print(f"\n{'='*80}")

ANALYZING AVAILABLE FLIGHT DATA

Total flights in dataset: 293,845
Flights from QRT headquarters: 19,652

Unique destinations reachable from QRT HQs: 891

Top 30 most connected destinations:
  MEL (AU): 532 routes - Melbourne Airport
  AMS (NL): 378 routes - Amsterdam Airport Schiphol
  BNE (AU): 301 routes - Brisbane Airport
  LHR (GB): 284 routes - Heathrow Airport
  FRA (DE): 277 routes - Frankfurt Airport
  SIN (SG): 265 routes - Singapore Changi Airport
  KUL (MY): 256 routes - Kuala Lumpur International Airport
  BKK (TH): 249 routes - Suvarnabhumi Airport
  CDG (FR): 245 routes - Paris Charles de Gaulle Airport
  DEL (IN): 212 routes - Indira Gandhi International Airport
  JFK (US): 195 routes - John F. Kennedy International Airport
  MAD (ES): 190 routes - Adolfo Suarez Madrid-Barajas Airport
  DOH (QA): 190 routes - Hamad International Airport
  LIS (PT): 179 routes - Lisbon Portela Airport
  HKG (HK): 175 routes - Hong Kong International Airport (Chek Lap Kok Airport)
  IST (

In [28]:
# =====================================================
# TRAIN MODEL WITH REAL FLIGHT DATA ONLY
# =====================================================
# Uses ONLY destinations that exist in training_data.csv
# with actual distance, CO2, and time data from real flights

import os
import json
import importlib
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from supersuit import pettingzoo_env_to_vec_env_v1, concat_vec_envs_v1

# Force reload the environment module
import sys
if 'travel_environment.env.travel_environment' in sys.modules:
    del sys.modules['travel_environment.env.travel_environment']
if 'travel_environment.env' in sys.modules:
    del sys.modules['travel_environment.env']
if 'travel_environment' in sys.modules:
    del sys.modules['travel_environment']

from travel_environment.env.travel_environment import TravelEnvironment

# Define the scenario with QRT headquarters as attendee locations
scenario = {
	"attendees": {
		"Mumbai": 2,
		"Shanghai": 3,
		"Hong Kong": 1,
		"Singapore": 2,
		"Sydney": 2
	},
	"availability_window": {
		"start": "2025-12-10T09:00:00Z",
		"end": "2025-12-15T17:00:00Z"
	},
	"event_duration": {
		"days": 0,
		"hours": 4
	}
}

# Create environment - it will dynamically build candidate locations from training data
parallel_env = TravelEnvironment(scenario)

print("="*80)
print("TRAINING WITH REAL FLIGHT DATA")
print("="*80)
print(f"✈️  Using ONLY destinations with actual flight data from QRT headquarters")
print(f"📍 QRT Headquarters (start locations): 13 cities")
print(f"🎯 Candidate destinations (from training data): {len(parallel_env.candidate_locations)} cities")
print(f"📊 Sample destinations: {', '.join(sorted(parallel_env.candidate_locations)[:10])}...")
print()

# Convert to vectorized environment
vec_env = pettingzoo_env_to_vec_env_v1(parallel_env)
vec_env = concat_vec_envs_v1(vec_env, num_vec_envs=1, num_cpus=1, base_class="stable_baselines3")

# Instantiate and train
model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log="./ppo_travel_tensorboard_realistic/")

print("Training for 200,000 timesteps with real flight data...")
model.learn(total_timesteps=200000)

# Save the model
model_path = "./trained_ppo_travel_model_realistic"
model.save(model_path)
print(f"\n✅ Model saved to: {model_path}")

# Evaluate
obs, info = parallel_env.reset()
actions = {}
for agent in parallel_env.agents:
    agent_obs = obs[agent]
    action, _states = model.predict(agent_obs, deterministic=True)
    actions[agent] = action

observations, rewards, dones, truncated, infos = parallel_env.step(actions)
parallel_env.render()

print("\n" + "="*80)
print("EVALUATION RESULTS")
print("="*80)
print(f"Attendees from QRT Headquarters: {scenario['attendees']}")
if parallel_env.agents:
    first_agent = parallel_env.agents[0]
    if first_agent in infos and 'chosen_city' in infos[first_agent]:
        chosen_city = infos[first_agent]['chosen_city']
        print(f"🎯 Optimal Event Location (based on real data): {chosen_city}")
print("="*80)

TRAINING WITH REAL FLIGHT DATA
✈️  Using ONLY destinations with actual flight data from QRT headquarters
📍 QRT Headquarters (start locations): 13 cities
🎯 Candidate destinations (from training data): 100 cities
📊 Sample destinations: Abeid Amani Karume, Aberdeen, Abha Regional, Adolfo Suarez Madrid-Barajas, Aksu, Al-Ahsa, Amsterdam Schiphol, Ankang Wulipu, Armidale, Ashgabat...

Using cpu device
Training for 200,000 timesteps with real flight data...
Logging to ./ppo_travel_tensorboard_realistic/PPO_1
------------------------------
| time/              |       |
|    fps             | 8935  |
|    iterations      | 1     |
|    time_elapsed    | 2     |
|    total_timesteps | 20480 |
------------------------------
---------------------------------------
| time/                   |           |
|    fps                  | 4843      |
|    iterations           | 2         |
|    time_elapsed         | 8         |
|    total_timesteps      | 40960     |
| train/                  |         

In [29]:
# =====================================================
# TEST REALISTIC MODEL - ONLY REAL FLIGHT DATA
# =====================================================

from stable_baselines3 import PPO
import sys
if 'travel_environment.env.travel_environment' in sys.modules:
    del sys.modules['travel_environment.env.travel_environment']
if 'travel_environment.env' in sys.modules:
    del sys.modules['travel_environment.env']
if 'travel_environment' in sys.modules:
    del sys.modules['travel_environment']

from travel_environment.env.travel_environment import TravelEnvironment

# Load the realistic model
model_realistic = PPO.load("./trained_ppo_travel_model_realistic")
print("✅ Realistic model loaded successfully!\n")

# Test scenarios
test_scenarios = [
    {
        "name": "Asia-Pacific QRT Offices",
        "attendees": {
            "Mumbai": 2,
            "Shanghai": 2,
            "Hong Kong": 2,
            "Singapore": 2,
            "Sydney": 2
        },
        "availability_window": {
            "start": "2025-12-10T09:00:00Z",
            "end": "2025-12-15T17:00:00Z"
        },
        "event_duration": {"days": 0, "hours": 4}
    },
    {
        "name": "European QRT Offices",
        "attendees": {
            "London": 2,
            "Paris": 2,
            "Zurich": 1,
            "Geneva": 1,
            "Budapest": 1
        },
        "availability_window": {
            "start": "2026-01-15T09:00:00Z",
            "end": "2026-01-20T17:00:00Z"
        },
        "event_duration": {"days": 0, "hours": 3}
    },
    {
        "name": "Global Mix",
        "attendees": {
            "London": 1,
            "Paris": 1,
            "Dubai": 2,
            "Mumbai": 2,
            "Singapore": 2,
            "Hong Kong": 1
        },
        "availability_window": {
            "start": "2026-03-10T09:00:00Z",
            "end": "2026-03-15T17:00:00Z"
        },
        "event_duration": {"days": 0, "hours": 4}
    }
]

print("="*80)
print("REALISTIC MODEL PREDICTIONS - BASED ON ACTUAL FLIGHT DATA")
print("="*80)

for test in test_scenarios:
    print(f"\n📍 Scenario: {test['name']}")
    print(f"   QRT Offices: {list(test['attendees'].keys())}")
    print(f"   Total Attendees: {sum(test['attendees'].values())}")
    
    # Create environment
    env = TravelEnvironment(test)
    print(f"   Available Destinations: {len(env.candidate_locations)} (with real flight data)")
    
    obs, info = env.reset()
    
    # Get predictions
    actions = {}
    for agent in env.agents:
        action, _ = model_realistic.predict(obs[agent], deterministic=True)
        actions[agent] = action
    
    # Step and get result
    observations, rewards, dones, truncated, infos = env.step(actions)
    
    if env.agents:
        chosen = infos[env.agents[0]]['chosen_city']
        print(f"   🎯 Recommended Location: {chosen}")
        
        # Show vote distribution
        votes = [env.candidate_locations[actions[agent]] for agent in env.agents]
        from collections import Counter
        vote_counts = Counter(votes)
        top_3 = vote_counts.most_common(3)
        print(f"   📊 Top 3 Choices: {', '.join([f'{city} ({count})' for city, count in top_3])}")
        
        # Calculate and show actual travel metrics for chosen location
        total_co2 = 0
        total_distance = 0
        travel_times = []
        
        for city, count in test['attendees'].items():
            co2 = env._get_co2_emissions(city, chosen)
            dist = env._get_distance(city, chosen)
            time = env._get_travel_time(city, chosen)
            total_co2 += co2 * count
            total_distance += dist * count
            travel_times.extend([time] * count)
        
        avg_travel_time = sum(travel_times) / len(travel_times) if travel_times else 0
        
        print(f"   💨 Total CO2 Emissions: {total_co2:.1f} kg")
        print(f"   ✈️  Average Travel Time: {avg_travel_time:.1f} hours")
        print(f"   📏 Total Distance: {total_distance:.0f} km")

print("\n" + "="*80)
print("✅ Realistic model recommendations based on ACTUAL flight data")
print("   - Uses real distances, CO2 emissions, and flight times")
print("   - Only suggests destinations with confirmed flight routes")
print("="*80)

✅ Realistic model loaded successfully!

REALISTIC MODEL PREDICTIONS - BASED ON ACTUAL FLIGHT DATA

📍 Scenario: Asia-Pacific QRT Offices
   QRT Offices: ['Mumbai', 'Shanghai', 'Hong Kong', 'Singapore', 'Sydney']
   Total Attendees: 10
   Available Destinations: 100 (with real flight data)
   🎯 Recommended Location: Takamatsu
   📊 Top 3 Choices: Takamatsu (4), Jaisalmer (2), Biju Patnaik (2)
   💨 Total CO2 Emissions: 3061.0 kg
   ✈️  Average Travel Time: 16.2 hours
   📏 Total Distance: 94422 km

📍 Scenario: European QRT Offices
   QRT Offices: ['London', 'Paris', 'Zurich', 'Geneva', 'Budapest']
   Total Attendees: 7
   Available Destinations: 100 (with real flight data)
   🎯 Recommended Location: Xi'an Xianyang
   📊 Top 3 Choices: Xi'an Xianyang (6), Biju Patnaik (1)
   💨 Total CO2 Emissions: 1498.9 kg
   ✈️  Average Travel Time: 17.4 hours
   📏 Total Distance: 50423 km

📍 Scenario: Global Mix
   QRT Offices: ['London', 'Paris', 'Dubai', 'Mumbai', 'Singapore', 'Hong Kong']
   Total Atten